# Datamine WFTREND Process Example

This notebook demonstrates how to configure and run the **`wftrend`** process wrapper in `dmstudio`.

## Process Description

## WFTREND

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

This process creates a DTM wireframe surface which can be extended beyond the data limits by a specified distance using a planar trend surface.

The data limits are detected automatically and the trend surface is only used beyond the limits of the actual data. The input data is a set of points which should be sorted.

### Input Files:

* **pointin** (Point Data):
  Input point data file.
  Required=Yes

### Output Files:

* **wiretr** (Wireframe Triangle):
  Output wireframe triangle file.
  Required=Yes

* **wirept** (Wireframe Points):
  Output wireframe point file.
  Required=Yes

* **pointou** (Point Data):
  Output point data file. This includes both the input point data, and the points which
  were extrapolated using the trend surface.
  Required=No

### Fields:

* **xpt** (Numeric : POINTIN):
  X field in input point data file.
  Default=XPT
  Required=Yes

* **ypt** (Numeric : POINTIN):
  Y field in input point data file.
  Default=YPT
  Required=Yes

* **zpt** (Numeric : POINTIN):
  Z field in input point data file.
  Default=ZPT
  Required=Yes

### Parameters:

* **extn**:
  The extension distance required beyond the data points for the triangulation.
  Range=0.1,+
  Values=Undefined
  Default=Undefined
  Required=Yes

* **gridinc**:
  Grid interval for the interpolated points beyond the limits of the original point data.
  Range=0.1,+
  Values=Undefined
  Default=Undefined
  Required=Yes

* **tr**:
  Flag to indicate whether the field in the wireframe triangle file should have different
  values depending on whether or not the triangle is in the extrapolated area: 0 = all
  triangles have colour defined by parameter 1 . 1 = triangles in the area covered by
  **POINTIN** data are coloured 1 and those in the extrapolated area are coloured 2.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Verify that the active project matches this folder (case-insensitive) to prevent writing files to the wrong place
active_folder = os.path.normpath(oScript.ActiveProject.Folder).lower()
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()

if active_folder != notebook_folder:
    raise RuntimeError(
        f"Active Datamine Project ({active_folder}) does not match this notebook's folder ({notebook_folder}).\n"
        "Please open the 'Project.rmproj' in this folder inside Datamine Studio RM first!"
    )
print("Active project verified successfully.")

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('wftrend')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine help database locally to this folder using a `t_` prefix. Paths are resolved relatively to ensure portability.

In [ ]:
# Step 3: Initialize example project dataset using relative paths
# Resolve relative path to repository's help database dynamically (4 levels up from subfolders)
repo_root = os.path.abspath(os.path.join(notebook_folder, '..', '..', '..', '..'))
help_db = os.path.join(repo_root, 'datamine_help', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

for filename in files_to_copy:
    src = os.path.join(help_db, filename)
    # Strip _vb_ prefix and prepend t_ for local usage
    local_name = "t_" + filename.replace("_vb_", "")
    dst = os.path.join(notebook_folder, local_name)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Initialized dataset: {local_name}")
    else:
        print(f"Warning: Source {filename} not found in help database.")

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute wftrend
print("Running wftrend...")
dm_cmd.wftrend(
    pointin_i='t_SurfacePointsPt',  # required
    wiretr_o='t_wftrend_out',  # required
    wirept_o='t_wftrend_out',  # required
    pointou_o='t_wftrend_out',  # required
    xpt_f='optional',  # required
    ypt_f='optional',  # required
    zpt_f='optional',  # required
    extn_p='required_val',  # required
    gridinc_p='required_val',  # required
    # tr_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("wftrend execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = os.path.join(notebook_folder, 't_wftrend_out.dmx')
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob(os.path.join(notebook_folder, "t_*.*"))
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    path = os.path.join(notebook_folder, f)
    if os.path.exists(path):
        try:
            os.remove(path)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = os.path.join(notebook_folder, '__pycache__')
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")